# 02 · Rotary position embeddings (RoPE)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/architecture/02_rope.ipynb) &nbsp; 📖 Concept deck: [RoPE](https://nilesh-salpe.github.io/visual-ai/architecture/03-rope/) &nbsp; ⏱️ Suggested time: **40 min**

**You will build:** RoPE in its rotation form and complex-number form, and tests that prove the key property: the attention score depends only on the distance between tokens.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def _rope_ref(x, pos, base=10000.0):
    """Interleaved-pair RoPE as in RoFormer: pairs (x0,x1), (x2,x3), ... rotate by pos * theta_i."""
    T, d = x.shape
    theta = base ** (-np.arange(0, d, 2) / d)            # [d/2]
    ang = np.asarray(pos, float)[:, None] * theta[None]  # [T, d/2]
    c, s = np.cos(ang), np.sin(ang)
    x0, x1 = x[:, 0::2], x[:, 1::2]
    out = np.empty_like(x, dtype=float)
    out[:, 0::2] = x0 * c - x1 * s
    out[:, 1::2] = x0 * s + x1 * c
    return out

## Task 1: RoPE as pairwise rotations

Split each vector of width $d$ into $d/2$ pairs $(x_{2i}, x_{2i+1})$. Pair $i$ has frequency
$$\theta_i = \text{base}^{-2i/d},\quad i = 0,\dots,d/2-1 \qquad(\text{base} = 10000 \text{ in RoFormer})$$
At position $m$, rotate pair $i$ by angle $m\theta_i$:
$$\begin{pmatrix}x'_{2i}\\x'_{2i+1}\end{pmatrix}=\begin{pmatrix}\cos m\theta_i & -\sin m\theta_i\\ \sin m\theta_i & \cos m\theta_i\end{pmatrix}\begin{pmatrix}x_{2i}\\x_{2i+1}\end{pmatrix}$$
RoPE is applied to **queries and keys only**, never to values, after the $W_q$ / $W_k$ projections.

> Llama's code pairs dimension $i$ with $i + d/2$ ("rotate half") instead of neighbours. That is the same idea with the dimensions permuted. Here we use the paper's neighbouring pairs.

In [ ]:
def rope(x, pos, base=10000.0):
    """x: [T, d] (d even); pos: [T] integer positions. Returns rotated [T, d]."""
    # TODO: theta = base ** (-arange(0, d, 2) / d); angle = pos[:, None] * theta
    #       rotate each pair (x[:, 0::2], x[:, 1::2]) by its angle
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_rope(f):
    r = np.random.default_rng(1); x = r.normal(size=(6, 16)); pos = np.array([0, 1, 2, 7, 100, 4096])
    y = f(x, pos)
    assert y.shape == x.shape, f"shape {y.shape} != {x.shape}"
    assert np.allclose(y[0], x[0]), "position 0 must be the identity (angle 0)"
    assert np.allclose(np.linalg.norm(y, axis=1), np.linalg.norm(x, axis=1)), "rotations must preserve each vector's length"
    assert np.allclose(y, _rope_ref(x, pos), atol=1e-9), "values differ: pair (x[2i], x[2i+1]) with theta_i = base**(-2i/d)"
    q, k = r.normal(size=(1, 16)), r.normal(size=(1, 16))
    s = lambda m, n: (f(q, [m]) @ f(k, [n]).T).item()
    assert np.isclose(s(5, 2), s(105, 102)) and np.isclose(s(5, 2), s(3, 0)), "score must depend only on m - n"

check("rope", _test_rope, rope)

## Task 2: The same thing with complex numbers

RoFormer writes RoPE as $f(x, m) = x\,e^{i m\theta}$: treat each pair as a complex number $x_{2i} + i\,x_{2i+1}$ and **multiply** by $e^{i m \theta_i}$.
This is how many production implementations do it (`torch.polar` in Meta's Llama code).
It also makes the relative-position proof one line:
$\langle q e^{im\theta}, k e^{in\theta}\rangle = \mathrm{Re}[q\,\bar k\, e^{i(m-n)\theta}]$, which depends only on $m-n$.

In [ ]:
def rope_complex(x, pos, base=10000.0):
    """Same contract as rope(), implemented with complex multiplication."""
    # TODO: z = x[:, 0::2] + 1j * x[:, 1::2]; z *= exp(1j * pos[:, None] * theta); write real/imag back into pairs
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_rope_complex(f):
    r = np.random.default_rng(2); x = r.normal(size=(5, 32)); pos = np.array([0, 3, 17, 512, 9999])
    y = f(x, pos)
    assert y.shape == x.shape, f"shape {y.shape} != {x.shape}"
    assert np.allclose(y, _rope_ref(x, pos), atol=1e-9), "complex form must match the rotation form exactly"

check("rope_complex", _test_rope_complex, rope_complex)

## Task 3: Attention scores with RoPE

Put it together for one head: given $q$ and $k$ of shape [T, d], apply RoPE at positions $0..T-1$, then return the **causal** score matrix $\frac{q'k'^\top}{\sqrt d}$ with $-\infty$ above the diagonal.
The test shifts the whole sequence to start at position 1000 and checks that **nothing changes**. This is the property absolute position embeddings lack.

In [ ]:
def rope_attention_scores(q, k, start=0, base=10000.0):
    """q, k: [T, d]. Positions are start..start+T-1. Returns causal scores [T, T] (-inf above the diagonal)."""
    # TODO: rotate q and k with rope(..., positions), scores = q' k'^T / sqrt(d), mask the future
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_rope_attention_scores(f):
    r = np.random.default_rng(3); q, k = r.normal(size=(6, 16)), r.normal(size=(6, 16))
    S0, S1 = f(q, k, 0), f(q, k, 1000)
    pos = np.arange(6); ref = _rope_ref(q, pos) @ _rope_ref(k, pos).T / 4
    tri = np.tril(np.ones((6, 6), bool))
    assert np.allclose(S0[tri], ref[tri]), "lower-triangular scores differ from reference (did you scale by 1/sqrt(d)?)"
    assert np.all(np.isneginf(S0[~tri])), "future positions must be -inf"
    assert np.allclose(S0[tri], S1[tri]), "shifting every position by 1000 must not change any score"

check("rope_attention_scores", _test_rope_attention_scores, rope_attention_scores)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: RoPE as pairwise rotations
def rope_solution(x, pos, base=10000.0):
    T, d = x.shape
    theta = base ** (-np.arange(0, d, 2) / d)
    ang = np.asarray(pos, float)[:, None] * theta[None, :]
    c, s = np.cos(ang), np.sin(ang)
    out = np.empty_like(x, dtype=float)
    out[:, 0::2] = x[:, 0::2] * c - x[:, 1::2] * s
    out[:, 1::2] = x[:, 0::2] * s + x[:, 1::2] * c
    return out

check("rope (reference)", _test_rope, rope_solution)

In [ ]:
# @title Solution for Task 2: The same thing with complex numbers
def rope_complex_solution(x, pos, base=10000.0):
    d = x.shape[1]
    theta = base ** (-np.arange(0, d, 2) / d)
    z = (x[:, 0::2] + 1j * x[:, 1::2]) * np.exp(1j * np.asarray(pos, float)[:, None] * theta[None, :])
    out = np.empty_like(x, dtype=float)
    out[:, 0::2], out[:, 1::2] = z.real, z.imag
    return out

check("rope_complex (reference)", _test_rope_complex, rope_complex_solution)

In [ ]:
# @title Solution for Task 3: Attention scores with RoPE
def rope_attention_scores_solution(q, k, start=0, base=10000.0):
    T, d = q.shape; pos = start + np.arange(T)
    s = rope_solution(q, pos, base) @ rope_solution(k, pos, base).T / np.sqrt(d)
    return np.where(np.tril(np.ones((T, T), bool)), s, -np.inf)

check("rope_attention_scores (reference)", _test_rope_attention_scores, rope_attention_scores_solution)

---
## Experiments
**Long-term decay and the frequency spectrum.** Take $q = k = $ a vector of ones and plot the score as the distance $m-n$ grows. The high-frequency pairs spin fast and average out, while the low-frequency pairs keep the score high for longer. RoFormer §3.4.3 shows that an upper bound on the score decays with distance. Then compare base 10,000 (Llama 2) with 500,000 (Llama 3): a larger base slows every rotation, so the decay stretches over a much longer range.

In [ ]:
d = 128; q = np.ones((1, d))
for base in [10_000, 500_000]:
    row = []
    for dist in [0, 1, 10, 100, 1_000, 10_000, 100_000]:
        row.append((rope_solution(q, [dist], base) @ rope_solution(q, [0], base).T).item() / d)
    print(f"base {base:>7,}: normalized score at distance 0,1,10,100,1k,10k,100k ->", ' '.join(f'{v:5.2f}' for v in row))
theta = 10_000 ** (-np.arange(0, d, 2) / d)
print("\nwavelength (tokens per full turn) of pair 0, 16, 32, 48, 63:", [f'{2 * np.pi / theta[i]:,.0f}' for i in [0, 16, 32, 48, 63]])

---
## Takeaways
* RoPE rotates each 2-D slice of $q$ and $k$ by an angle proportional to the position, with a different speed per slice.
* Because both are rotated, the dot product sees only the **difference** of angles, so attention depends on relative position. No position vector is added to the residual stream.
* Wavelengths range from about 6 tokens (pair 0) to about 54,000 tokens (last pair) at base 10,000. Long-context work (deck 07) is about what happens to the slow pairs beyond the training length.